# 3. 어텐션 메커니즘 구현하기

지금까지 LLM 훈련을 위해 텍스트를 개별 단어와 부분단어 토큰으로 나누어 입력 데이터를 준비하는 방법을 배우고, LLM에 사용할 수 있도록 이를 벡터 표현으로 인코딩했다.

이 장에서는 네 가지 버전의 어텐션 메커니즘을 차례대로 구축하되, 이전에 구현한 것에 새로운 기능을 추가하는 식으로 만든다.
1. 간소화된 셀프 어텐션: 간소화된 셀프 어텐션으로 개괄적인 아이디어를 소개한다.
2. 셀프 어텐션: LLM 메커니즘의 기반을 형성하는 훈련 가능한 가중치가 있는 셀프 어텐션
3. 코잘 어텐션: 모델이 시퀀스에서 현재와 이전 입력에만 주의를 기울이기 위해 LLM에서 사용하는 셀프 어텐션이며 텍스트 생성 시 시간 순서를 보장한다.
4. 멀티헤드 어텐션: 셀프 어텐션과 코잘 어텐션을 확장하여 모델이 동시에 다른 표현 공간의 정보에 주의를 기울일 수 있게 만든다.

## 3.1 긴 시퀀스 모델링의 문제점
LLM의 핵심 요소인 셀프 어텐션 메커니즘으로 들어가기 전에 LLM 시대 이전에 어텐션 메커니즘을 사용하지 않던 구조가 가졌던 문제를 생각해 보자. 한 언어에서 다른 언어로 텍스트를 번역하는 언어 번역 모델을 만든다고 가정해 보자. source 언어와 target 언어의 문법 구조가 다르기 때문에 텍스트를 한 단어씩 번역할 수 없다.

이 문제를 해결하기 위해 심층 신경망에는 인코더(encoder)와 디코더(decoder)라는 2개의 서브모듈을 사용한다. 인코더가 먼저 전체 텍스트를 읽고 처리한 후 디코더가 번역된 텍스트를 생성한다.

트랜스포머가 개발되기 전에는 순환 신경망(RNN)이 언어 번역에서 가장 인기 있는 인코더-디코더 구조였다. RNN은 이전 스텝의 출력이 현재 스텝의 입력으로 사용되는 신경망으로, 텍스트와 같은 순차 데이터에 잘 맞는다.

인코더-디코더 RNN에서는 인코더가 입력 텍스트를 받아 순차적으로 처리한다. 인코더는 각 단계마다 은닉 상태(은닉 층의 내부 값)를 업데이트하며, 최종 은닉 상태로 입력 시퀀스의 전체 의미를 포착한다. 그런 다음 디코더는 이 최종 은닉 상태를 받아 한 번에 한 단어씩 번역된 문장을 생성한다. 디코더도 매 스텝마다 은닉 상태를 업데이트하며 이를 통해 다음 단어 예측에 필요한 문맥을 다음 스텝으로 전달한다.

인코더-디코더 RNN의 큰 제약 사항은 디코딩 단계에서 RNN이 인코더의 이전 은닉 상태를 참조할 수 없다는 것이다. 결과적으로 모든 관련된 정보가 포함된 최종 은닉 상태에만 의존하게 된다. 이로 인해 맥락을 놓칠 수 있다. 특히 멀리 떨어진 단어에 의존성이 있는 복잡한 문장의 경우가 그렇다.

## 3.2 어텐션 메커니즘으로 데이터 의존성 포착하기
RNN은 짧은 문장을 번역하는 데는 잘 동작하지만 입력에 있는 이전 단어를 직접 참조하지 못하기 때문에 긴 텍스트는 잘 번역하지 못한다. 이 방식의 주요 단점 중 하나는 RNN이 인코딩된 전체 입력을 하나의 은닉 상태에 저장해서 디코더에게 전달해야 한다는 것이다.

이로 인해 2014년 RNN을 위한 바흐다나우 어텐션 메커니즘이 개발되었다. 이 메커니즘은 인코더-디코더 RNN을 수정하여 디코딩 단계마다 디코더가 선택적으로 입력 시퀀스의 서로 다른 부분을 참고할 수 있다.

흥미롭게도 불과 3년 후에 RNN 구조가 자연어 처리를 위한 심층 신경망을 구축하는 데 필수적이지 않다는 것을 발견했고 원본 **트랜스포머** 구조가 등장했다. 트랜스포머에는 바흐다나우 어텐션 메커니즘에서 영감을 받은 셀프 어텐션 메커니즘이 포함되어 있다.

셀프 어텐션 메커니즘에서는 시퀀스의 표현을 계산할 때 입력 시퀀스에 있는 각 위치가 동일 시퀀스에 있는 다른 모든 위치와의 관련성을 고려하거나 주의를 기울일 수 있다. 셀프 어텐션은 트랜스포머 구조 기반의 GPT 시리즈와 같은 현대 LLM의 핵심 구성 요소이다.

## 3.3 셀프 어텐션으로 입력의 서로 다른 부분에 주의 기울이기
이제 셀프 어텐션 메커니즘의 내부 작동 방식을 알아보고 코드로 구현해보자. 셀프 어텐션은 트랜스포머 구조를 바탕으로 하는 모든 LLM의 기반이 된다.



> 셀프 어텐션에서 '셀프'는 하나의 입력 시퀀스에 있는 서로 다른 위치의 원소 사이에서 어텐션 가중치를 계산하는 방식을 의미한다. 입력 자체 안에 있는 여러 부분(예를 들어 문장 안의 단어나 이미지 안에 있는 픽셀) 사이의 관계와 의존성을 평가하고 학습한다.

처음엔 셀프 어텐션이 복잡하게 보일 수 있기 때문에 간소화된 버전을 먼저 알아보고, LLM에서 사용되는 훈련 가능한 가중치를 가진 셀프 어텐션 메커니즘을 구현해 보자.

### 3.3.1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션 메커니즘
$\mathbf{x}$는 입력 시퀀스이며, $\mathbf{x}^{(1)}$에서 $\mathbf{x}^{(T)}$ 까지 표현된 $T$개의 원소로 구성된다. 이 시퀀스는 일반적으로 (문장과 같은) 텍스트를 나타내며 이미 토큰 임베딩으로 변환되어 있다.

예를 들어 "Your journey starts with one step"이란 텍스트를 생각해보자. 이 경우 $\mathbf{x}^{(1)}$에 해당하는 시퀀스의 각 원소는 "Your"와 같은 특정 토큰을 표현하는 d 차원 임베딩 벡터이다.

셀프 어텐션에서는 입력 시퀀스에 있는 각 원소 $\mathbf{x}^{(i)}$에 대한 문맥 벡터 $\mathbf{z}^{(i)}$를 계산하는 것이 목표이다. 문맥 벡터(context vector)는 정보가 풍부한 임베딩 벡터로 생각할 수 있다.

토큰 "journey"에 해당하는 두 번째 입력 원소 $\mathbf{x}^{(2)}$의 임베딩 벡터와 문맥 벡터 $\mathbf{z}^{(2)}$를 생각해보자. 문맥 벡터 $\mathbf{z}^{(2)}$는 $\mathbf{x}^{(2)}$와 다른 모든 입력($\mathbf{x}^{(1)}$ ~ $\mathbf{x}^{(T)}$) 사이의 정보를 담은 임베딩이다.

문맥 벡터는 셀프 어텐션에서 매우 중요한 역할을 한다. 문맥 벡터의 목적은 (문장과 같은) 입력 시퀀스에 있는 다른 모든 원소의 정보를 통합해 이 시퀀스에 있는 각 원소의 표현을 풍부하게 만드는 것이다. 이것이 LLM의 핵심이며, 문장에서 다른 단어 사이의 관계와 관련성을 이해하기 위해 LLM에 반드시 필요하다.

다음과 같이 3차원 벡터로 임베딩된 입력 시퀀스가 있다고 가정해보자.



In [ ]:
import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # journey (x^2)
     [0.57, 0.85, 0.64], # starts (x^3)
     [0.22, 0.58, 0.33], # with (x^4)
     [0.77, 0.25, 0.10], # one (x^5)
     [0.05, 0.80, 0.55]] # step (x^6)
)

셀프 어텐션을 구현하는 첫 번째 단계는 어텐션 점수(attention score)라고 부르는 중간값 $\omega$를 계산하는 것이다.

어텐션 점수는 쿼리 $\mathbf{x}^{(2)}$와 다른 모든 입력 토큰 사이의 점곱으로 결정한다.

In [ ]:
query = inputs[1] # 두 번째 입력 토큰을 쿼리 토큰으로 사용한다.
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

다음 단계에서는 앞서 계산한 어텐션 점수를 정규화한다. 정규화를 하는 목적은 어텐션 가중치의 합이 1이 되도록 하기 위해서이다. 정규화를 하면 해석하기 용이하고 LLM을 훈련할 때 안전성을 유지하는 데 도움이 된다.

In [ ]:
attn_weights_2_temp = attn_scores_2 / attn_scores_2.sum()
print("어텐션 가중치: ", attn_weights_2_temp)
print("합: ", attn_weights_2_temp.sum())

실제로는 softmax 함수를 사용하여 정규화하는 것이 더 일반적이고 권장된다. 다음 코드는 어텐션 점수를 정규화하는 softmax 함수의 기본적인 구현이다.

In [ ]:
def softmax_naive(x):
  return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("어텐션 가중치: ", attn_weights_2_naive)
print("합: ", attn_weights_2_naive.sum())

또한 softmax 함수는 어텐션 가중치가 항상 양수가 되도록 보장한다. 이렇게 하면 출력을 확률이나 상대적인 중요도로 해석할 수 있으며, 가중치가 높을수록 중요도가 높다.

실전에서는 광범위하게 성능 최적화가 된 파이토치의 softmax 함수를 사용하는 것이 권장된다.

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("어텐션 가중치: ", attn_weights_2)
print("합: ", attn_weights_2.sum())

정규화된 어텐션 가중치를 계산했으니, 이제 마지막 단계를 수행할 준비가 됐다.

임베딩된 입력 토큰 $\mathbf{x}^{(i)}$와 각 토큰에 해당하는 어텐션 가중치를 곱한 후 모두 더해서 문맥 벡터 $\mathbf{z}^{(2)}$를 계산한다. 따라서 문맥 벡터 $\mathbf{z}^{(2)}$는 모든 입력 벡터의 가중치 합이 되며, 각 입력 벡터와 어텐션 가중치를 곱하여 구한다.

In [ ]:
query = inputs[1] # 두 번째 입력 토큰이 쿼리이다.
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

다음으로 문맥 벡터 계산 과정을 일반화하여 모든 문맥 벡터를 동시에 계산한다.

### 3.3.2 모든 입력 토큰에 대해 어텐션 가중치 계산하기

지금까지 두 번째 입력 토큰에 대한 어텐션 가중치와 문맥 벡터를 계산했다. 이제 이 계산을 확장하여 모든 입력에 대한 어텐션 가중치와 문맥 벡터를 계산해보자.

In [ ]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
  for j, x_j in enumerate(inputs):
    attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

텐서의 각 원소는 모든 입력 사이의 어텐션 점수를 나타낸다.

앞서 어텐션 점수를 계산할 때는 파이썬의 for 루프를 사용했다. 하지만 for 루프는 느리기 때문에 대신 행렬 곱셈을 사용해 동일한 결과를 얻을 수 있다.

In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)

다음으로 각 행의 값을 모두 더하면 1이 되도록 정규화한다.

In [ ]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

파이토치의 torch.softmax와 같은 함수에서 dim 매개변수는 계산이 수행될 입력 텐서의 차원을 지정한다. dim=-1로 지정하면 softmax 함수가 attn_scores 텐서의 마지막 차원을 따라 정규화를 수행한다. attn_scores가 2차원 텐서(에를 들어 [rows, columns] 크기인 텐서)라면, 열을 따라서 각 행의 값을 모두 더해 1이 되도록 정규화한다.

각 행의 값이 모두 더해서 1이 되는지 확인해보자.

In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("두 번째 행의 합: ", row_2_sum)
print("모든 행의 합: ", attn_weights.sum(dim=-1))

마지막 단계에서는 이 어텐션 가중치와 입력을 행렬 곱셈하여 모든 문맥 벡터를 계산한다.

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

3.3.1절에서 계산한 문맥 벡터 $\mathbf{z}^{(2)}$와 두 번째 행을 비교하여 계산이 올바르게 수행되었는지 확인해보자.

In [ ]:
print("앞서 계산한 두 번째 문맥 벡터: ", context_vec_2)

지금까지 간단한 셀프 어텐션 메커니즘을 코드로 구현해 보았다. 다음으로 LLM이 데이터로부터 학습하고 특정 작업에서 성능을 향상할 수 있도록 훈련 가능한 가중치를 추가해보자.

## 3.4 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기

다음 단계로 원본 트랜스포머 구조는 물론 GPT와 대부분의 다른 LLM에서 사용되는 셀프 어텐션 메커니즘을 구현한다. 이 셀프 어텐션 메커니즘을 scaled dot-product이라 부른다.

가장 큰 차이점은 모델 훈련 과정에서 업데이트되는 가중치 행렬이 추가된 것이다. 이 훈련 가능한 가중치 행렬을 통해 모델(구체적으로 모델 안의 어텐션 모델)이 '좋은' 문맥 벡터를 생성하는 방법을 배울 수 있기 때문에 매우 중요하다.

이 셀프 어텐션 메커니즘을 2개의 하위 절로 나누어 다루어 보겠다. 첫째, 이전처럼 단계별로 어텐션 메커니즘을 코드로 구현한다. 둘째, LLM 구조에 이식하기 위해 이 코드를 간결한 파이썬 클래스로 변환한다.

### 3.4.1 단계별로 어텐션 가중치 계산하기
훈련 가능한 가중치 행렬 3개, 즉 $\mathbf{W_q}, \mathbf{W_k}, \mathbf{W_v}$를 추가하여 셀프 어텐션 메커니즘을 단계별로 구현한다. 이 3개의 행렬을 사용해 임베딩된 입력 토큰 $\mathbf{x}^{(i)}$를 각각 쿼리(query), 키(key), 값(value) 벡터로 투영한다.

예시를 위해 먼저 하나의 문맥 벡터 $\mathbf{z}^{(2)}$를 계산해보자. 그런 다음 이 코드를 수정하여 모든 문맥 벡터를 계산한다.

In [ ]:
x_2 = inputs[1] # 두 번째 입력 원소
d_in = inputs.shape[1] # 입력 임베딩 크기, d_in=3
d_out = 2 # 출력 임베딩 크기, d_out=2

GPT와 같은 모델에서는 입력 차원과 출력 차원이 일반적으로 동일하지만 계산 과정을 이해하기 쉽도록 입력 차원(d_in=3)과 출력 차원(d_out=2)을 달리하였다.

그런 다음 3개의 가중치 행렬 $\mathbf{W_q}, \mathbf{W_k}, \mathbf{W_v}$를 초기화한다.

In [ ]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

여기서는 출력을 간단하게 만들려고 requires_grad=False로 지정했지만, 모델 훈련 시에는 requires_grad=True로 지정하여 훈련 과정에서 가중치 행렬을 업데이트해야 한다.

그런 다음 쿼리, 키, 값 벡터를 계산한다.

In [ ]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

임시로 하나의 문맥 벡터 $\mathbf{z}^{(2)}$만 계산하지만, 쿼리 $\mathbf{q}^{(2)}$에 대한 어텐션 가중치를 계산하기 위해서는 모든 입력 원소에 대한 키와 값 벡터가 필요하다.

행렬 곱셈을 통해 모든 키와 값을 구할 수 있다.

In [ ]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape: ", keys.shape)
print("values.shape: ", values.shape)

두 번째 단계에서는 어텐션 점수를 계산한다.

먼저 어텐션 점수 $\omega_{22}$를 계산해본다.

In [ ]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

행렬 곱셈으로 이 계산을 일반화하여 모든 어텐션 점수를 계산할 수 있다.

In [ ]:
attn_scores_2 = query_2 @ keys.T # 주어진 쿼리에 대한 모든 어텐션 점수
print(attn_scores_2)

이제 어텐션 점수에서 어텐션 가중치를 구해보자. 어텐션 점수를 softmax 함수로 정규화하여 어텐션 가중치를 구한다. 하지만 어텐션 점수를 키의 임베딩 차원의 제곱근으로 나눈다.

In [ ]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

마지막 단계에서 각각의 값 벡터와 어텐션 가중치를 곱한 후 모두 더하여 문맥 벡터를 구한다.

In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

지금까지 하나의 문맥 벡터 $\mathbf{z}^{(2)}$를 계산했다. 이 코드를 일반화하여 입력 시퀀스의 모든 문맥 벡터 $\mathbf{z}^{(1)}$에서 $\mathbf{z}^{(T)}$까지 계산해보자.

### 3.4.2 셀프 어텐션 파이썬 클래스 구현하기

In [ ]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in, d_out))
    self.W_key = nn.Parameter(torch.rand(d_in, d_out))
    self.W_value = nn.Parameter(torch.rand(d_in, d_out))

  def forward(self, x):
    keys = x @ self.W_key
    queries = x @ self.W_query
    values = x @ self.W_value
    attn_scores = queries @ keys.T # omega
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
    )
    context_vec = attn_weights @ values
    return context_vec

이 파이토치 코드에서 SelfAttention_v1 클래스는 파이토치 모델의 기본 구성 요소인 nn.Module 클래스를 상속한다. nn.Module 클래스는 층을 생성하고 관리하기 위해 필요한 기능을 제공한다.

\_\_init__ 메서드는 쿼리, 키, 값을 위한 훈련 가중치 행렬 (W_query, W_key, W_value)을 초기화한다. 이 행렬을 사용해 입력을 차원 d_in에서 출력 차원 d_out으로 변환한다.

forward 메서드롤 정방향 계산(forward pass)를 수행하는 동안 어텐션 점수(attention_scores)를 계산하고 softmax 함수로 이 점수를 정규화한다. 그런 다음 정규화된 어텐션 점수로 값에 가중치를 부여하여 문맥 벡터를 만든다.

이 클래스는 다음과 같이 사용할 수 있다.

In [ ]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

셀프 어텐션은 훈련 가능한 가중치 행렬 $\mathbf{W_q}, \mathbf{W_k}, \mathbf{W_v}$를 사용한다. 이런 행렬이 입력 데이터를 각각 쿼리, 키, 값으로 변형시키며, 이는 어텐션 메커니즘에서 매우 중요한 부분이다.

SelfAttention_v1 구현을 파이토치의 nn.Linear 층을 사용하도록 개선할 수 있다. 실제로 nn.Linear 층은 bias unit을 사용하지 않는 경우 행렬 곱셈과 동일한 연산을 수행한다. 또한 nn.Linear 층은 최적화된 가중치 초기화 방법을 사용할 수 있어 모델 훈련을 안정적이고 효율적으로 만들기 때문에 수동으로 nn.Parameter(torch.rand(...))을 사용하는 것에 비해 장점이 크다.

In [ ]:
class SelfAttention_v2(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)
    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
    )
    context_vec = attn_weights @ values
    return context_vec

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

그런 다음 코잘 어텐션과 멀티 헤드 어텐션으로 셀프 어텐션 메커니즘을 개선한다. 코잘 어텐션은 어텐션 메커니즘을 수정하여 모델이 시퀀스의 미래 정보를 참조하지 못하도록 막는다. 이는 각각의 단어 예측을 이전 단어에만 의존해야 하는 언어 모델링 같은 작업에 매우 중요하다.

멀티 헤드 어텐션은 어텐션 메커니즘을 여러 개의 '헤드(head)'로 나눈다. 각 헤더는 데이터에셔 여러 다른 측면을 학습하여 모델이 동시에 다른 위치에 있는 다른 표현 공간의 정보에 주의를 기울일 수 있다. 이렇게 하면 복잡한 작업에서 모델 성능이 향상된다.

## 3.5 코잘 어텐션으로 미래의 단어를 감추기
많은 LLM 작업에서 현재 위치 이전의 토큰만 참조하여 시퀀스의 다음 단어를 예측하도록 셀프 어텐션 메커니즘을 사용해야 한다. **마스크드 어텐션(masked attention)** 이라고도 부르는 코잘 어텐션은 셀프 어텐션의 특별한 형태이다. 이 모델은 메커니즘이 주어진 토큰으로 어텐션 점수를 계산할 때 시퀀스의 이전 입력과 현재 입력만 참조하도록 만든다. 이와 다르게 기본적인 셀프 어텐션 메커니즘은 한 번에 입력 시퀀스 전체를 모두 사용한다.

기본 셀프 어텐션 메커니즘을 수정하여 **코잘 어텐션(causal attention)** 메커니즘을 만들어보자. GPT와 같은 LLM에서는 각 토큰을 처리할 때 입력 텍스트에서 현재 토큰 다음에 오는 미래 토큰을 마스킹한다. 주대각선 위의 어텐션 가중치를 마스킹하고 마스킹하지 않은 어텐션 가중치를 정규화하여 각 행의 합이 1이 되게 만든다.

### 3.5.1 코잘 어텐션 마스크 적용하기
다음 단계는 코잘 어텐션 마스크를 코드로 구현하는 것이다.

>(정규화되지 않은) 어텐션 점수 -> (정규화된) 어텐션 가중치 -> (정규화되지 않은) 마스킹된 어텐션 점수 -> (정규화된) 마스킹된 어텐션 가중치

먼저 이전에 했던 것처럼 소프트맥스 함수를 사용해 어텐션 가중치를 계산한다.



In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

그런 다음 파이토치의 tril 함수로 주대각선 위의 값이 0인 마스크를 만든다.

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

이제 이 마스크와 어텐션 가중치를 곱해서 주대각선 위의 값을 0으로 만든다.

In [ ]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

마지막으로 이 어텐션 가중치를 합이 1이 되도록 다시 정규화한다. 각 행의 합으로 행의 원소를 나누면 된다.

In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple/row_sums
print(masked_simple_norm)

이 시점에서 코잘 어텐션 구현을 마무리할 수 있지만 더 개선할 수 있다. softmax 함수의 수학적 성질을 사용해 마스킹된 어텐션 가중치를 더 적은 단계에서 효율적으로 계산할 수 있다.



> (정규화되지 않은) 어텐션 점수에서 주대각선 위의 값을 $-\infty$으로 마스킹한다. -> (정규화되지 않은) 마스킹된 어텐션 점수에서 softmax 함수를 적용하여 -> (정규화된) 마스킹된 어텐션 가중치를 만든다.

softmax 함수는 입력을 확률분포로 변환한다. 한 행에 음의 무한대 값이 있으면 softmax는 해당 값을 0으로 만든다. 수학적으로 $e^{-\infty}$는 0에 수렴하기 때문이다.



In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

이제 남은 일은 이 마스킹된 결과에 softmax 함수를 적용하는 것이다.

In [ ]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

이 어텐션 가중치를 사용해 context_vec = attn_weights @ values와 같이 문맥 벡터를 계산할 수 있다. 하지만 LLM을 훈련할 때 과대적합을 줄이기 위해 코잘 어텐션 메커니즘에 필요한 추가 작업이 있다.

### 3.5.2 드롭아웃으로 어텐션 가중치에 추가적으로 마스킹하기
딥러닝에서 드롭아웃(dropout)은 훈련 중에 은닉층의 유닛을 랜덤하게 선택하여 해당 유닛의 출력을 무시하는 기법이다. 모델이 은닉층의 특정 유닛에 과도하게 의존하지 않도록 하여 과대적합을 막는 데 도움이 된다. 드롭아웃은 훈련 중에만 사용되며 그 이후에는 비활성화된다는 점이 중요하다.

GPT와 같은 모델을 포함해서 트랜스포머 구조에서 일반적으로 어텐션 메커니즘에 드롭아웃이 적용되는 곳은 두 군데이다. 어텐션 가중치를 계산 후 또는 값 벡터에서 어텐션 가중치를 적용한 후이다. 여기서는 어텐션 가중치를 계산한 후에 드롭아웃을 적용한다. 이 방식이 일반적으로 더 널리 사용되기 때문이다.

다음 코드에서는 어텐션 가중치의 절반을 마스킹하기 위해 드롭아웃 비율을 50%로 지정한다. (나중에 GPT 모델을 훈련할 때는 0.1 또는 0.2와 같은 낮은 드롭아웃 비율을 사용한다.)

간단한 예시를 위해 6 x 6 텐서에 파이토치 드롭아웃 층을 적용해본다.

In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6) # 1로 채워진 행렬을 만든다.
print(dropout(example))

어텐션 가중치 행렬에 50%의 비율로 드롭아웃을 적용하면 행렬에 있는 원소 절반이 랜덤하게 0으로 바뀐다. 삭제된 값을 남은 원소들로 보상하기 위해 행렬에서 남은 원소의 값을 1/0.5 -> 2배로 늘린다. 이런 보상은 전반적인 어텐션 가중치의 균형을 유지하는 데 중요하다. 이를 통해 훈련과 추론 시에 어텐션 메커니즘이 미치는 평균적인 영향을 균일하게 만든다.

그럼 이제 어텐션 가중치 행렬에 드롭아웃을 적용해보자.

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

어텐션 가중치 행렬의 원소가 추가적으로 0으로 바뀌었고, 남은 원소의 값은 증가되었다.

코잘 어텐션과 드롭아웃 마스크를 이해했으니 이제 간단한 파이썬 클래스를 만들어보자. 이 클래스에서 두 기법을 효율적으로 구현한다.

### 3.5.3 코잘 어텐션 클래스 구현하기
3.4절에서 만든 SelfAttention 클래스에 코잘 어텐션과 드롭아웃 기능을 추가한다. 그런 다음 이 클래스를 바탕으로 최종적으로 구현할 어텐션 클래스인 멀테 헤드 어텐션(multi-head attention)을 개발한다.

하지만 시작하기 전에 1개 이상의 입력으로 구성된 배치를 처리할 수 있는지 확인해야 한다.

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

이렇게 하면 2개의 입력 텍스트로 구성된 3차원 텐서가 된다. 각 입력 텍스트는 6개의 토큰을 가지고 있고 각 토큰은 3차원 임베딩 벡터이다.

다음에 나오는 CausalAttention 클래스는 코잘 마스크와 드롭아웃 마스크를 추가한 것을 제외하면 앞서 구현한 SelfAttention 클래스와 비슷하다.

In [ ]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length,
               dropout, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
    self.dropout = nn.Dropout(dropout) # SelfAttention_v1 클래스와 달리 드롭아웃 층을 추가한다.
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length),
        diagonal=1)
    ) # register_buffer 메서드 호출도 추가된다.

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1, 2) # 첫 번째 차원인 배치 차원은 그대로 유지하면서, 두번째 차원과 세번쨰 차원을 바꾼다.
    attn_scores.masked_fill_(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
    )
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
    )
    attn_weights = self.dropout(attn_weights)

    context_vec = attn_weights @ values
    return context_vec

이제 __init()__ 메서드에 self.register_buffer() 호출이 추가되었다. 파이토치에서 모든 경우에 register_buffer를 꼭 사용해야 하는 것은 아니지만 여기에서는 몇 가지 이점이 있다. 예를 들어 LLM에서 CausalAttention 클래스를 사용할 때 register_buffer() 메서드에 지정한 텐서를 모델과 함께 적절한 장치(CPU 또는 GPU)로 자동으로 이동시킨다. 이는 LLM 훈련할 때와 관련이 있다. 즉, 장치가 동일하지 않다는 오류를 피하기 위해 텐서가 모델 파라미터와 같은 장치에 있는지 수동으로 확인할 필요가 없다.

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

다음으로 이 개념을 확장하여 여러 개의 코잘 어텐션을 병렬로 구성한 멀티 헤드 어텐션 모듈을 구현해 보자.

## 3.6 싱글 헤드 어텐션을 멀티 헤드 어텐션으로 확장하기
마지막 단계로 앞서 구현한 코잘 어텐션 클래스를 멀티 헤드 버전으로 확장하자. 이를 **멀티 헤드 어텐션**이라 부른다.

'멀티 헤드'란 용어는 어텐션 메커니즘을 독립적으로 동작하는 여러 개의 '헤드'로 나누었ㄷ는 의미이다. 이런 맥락에서 단일 코잘 어텐션 모듈을 싱글 헤드 어텐션으로 생각할 수 있다.

코잘 어텐션에서 멀티 헤드 어텐션으로 확장하는 문제를 다음과 같이 접근해 보자. 먼저 CausalAttention 모듈을 여러 개 쌓아서 멀티 헤드 어텐션 모듈을 구축해 보자. 그런 다음 동일한 멀티 헤드 어텐션 모듈을 좀 더 복잡하지만 더 효율적인 방식으로 구현한다.

### 3.6.1 여러 개의 싱글 헤드 어텐션 층 쌓기
실제로 멀티 헤드 어텐션을 구현하기 위해 각자 고유한 가중치를 가진 셀프 어텐션 메커니즘을 여러 개 만들고 출력을 합칠 수 있다. 셀프 어텐션 메커니즘을 여러 개 만들면 계산량이 늘어나지만, 트랜스포머 기반 LLM과 같은 모델의 복잡한 패턴 인식에는 매우 중요하다.

앞서 구현한 CausalAttention 모듈을 여러 개 쌓아 간단한 MultiHeadAttentionWrapper 클래스를 구현할 수 있다.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in, d_out, context_length,
               dropout, num_heads, qkv_bias=False):
    super().__init__()
    self.heads = nn.ModuleList(
        [CausalAttention(
            d_in, d_out, context_length, dropout, qkv_bias
        ) for _ in range(num_heads)
        ]
    )

  def forward(self, x):
    return torch.cat([head(x) for head in self.heads], dim=-1)

MultiHeadAttentionWrapper를 사용하려면 어텐션 헤드의 개수(num_heads)를 지정한다. num_heads가 2라면 두 벌의 문맥 벡터 행렬을 얻는다. 각각의 문맥 벡터 행렬에서 행은 토큰에 대한 문맥 벡터에 해당한다. 각 문맥 벡터의 열은 d_out=2로 지정한 임베딩 차원에 해당한다. 이 문맥 벡터 행렬을 열 차원을 따라 연결한다. 임베딩 차원이 2인 2개의 어텐션 헤드가 있으므로 최종 임베딩 차원은 2 x 2 = 4이다.

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1] # 토큰 개수
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

지금까지 싱글 헤드 어텐션 모듈 여러 개를 결합하는 MultiHeadAttentionWrapper 클래스를 구현했다. 하지만 정방향 계산에서 [head(x) for head in self.heads]와 같이 순차적으로 계산한다. 헤더를 병렬로 처리할 수 있도록 이 구현을 개선할 수 있다. 한 가지 방법으 행렬 곱셈을 통해 모든 어텐션 출력을 동시에 계산하는 것이다.

### 3.6.2 가중치 분할로 멀티 헤드 어텐션 구현하기
지금까지 여러 개의 싱글 헤드 어텐션 모듈을 쌓아 멀티 헤드 어텐션을 만드는 MultiHeadAttentionWrapper를 구현했다. 이 클래스는 CausalAttention 클래스의 객체를 여러 개 만들어 연결하는 식으로 구현되었다.

MultiHeadAttentionWrapper와 CausalAttention 클래스 2개를 별도로 관리하는 대신에 이 개념을 하나의 MultiHeadAttention 클래스로 결합할 수 있다. 또한 MultiHeadAttentionWrapper와 CausalAttention 클래스를 합치는 것 외에도 멀티 헤드 어텐션을 효율적으로 구현하기 위해 몇 가지 수정 사항을 추가한다.

MultiHeadAttentionWrapper에서는 개별 어텐션 헤드를 나타내는 CausalAttention 객체가 담긴 리스트(self.heads)를 만들어 여러 개의 헤드를 구현했다. CausalAttention 클래스가 독립적으로 어텐션 메커니즘을 수행한 후 각 헤드의 결과를 연결한다. 반면 MultiHeadAttention 클래스는 하나의 클래스 안에 멀티 헤드 기능을 통합한다. 선형 투영된 쿼리, 키, 값 텐서의 크기를 변경하는 방법을 사용하여 입력을 여러 개의 헤드로 나눈다. 그런 다음 어텐션을 계산한 후 각 헤드의 결과를 결합한다.

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out,
               context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert(d_out % num_heads ==0) , \
        "d_out은 num_heads로 나누어 떨어져야 합니다"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_out, d_out)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    # 텐서 크기: (b, num_tokens, d_out)
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    # num_heads 차원을 추가함으로써 암묵적으로 다음 행렬을 분할한다.
    # 그런 다음 마지막 차원을 num_heads에 맞춰 채운다.
    # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    # (b, num_tokens, num_heads, head_dim) 크기를
    # (b, num_heads, num_tokens, head_dim)로 바꾼다.
    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)

    # 각 헤드에 대해 점곱을 수행한다.
    attn_scores = queries @ keys.transpose(2, 3)
    # 토큰 개수로 마스크를 자른다.
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    # 마스크를 사용해 어텐션 점수를 채운다.
    attn_scores.masked_fill_(mask_bool, -torch.inf)

    attn_weigths = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weigths)

    # 텐서 크기: (b, num_tokens, num_heads, head_dim)
    context_vec = (attn_weights @ values).transpose(1, 2)

    # 헤드를 결합한다. self.d_out = self.num_heads * self.head_dim
    context_vec = context_vec.contiguous().view(
        b, num_tokens, self.d_out
    )

    # 선형 투영을 추가한다.
    context_vec = self.out_proj(context_vec)
    return context_vec

MultiHeadAttentionWrapper에서는 싱글 헤드 어텐션 층을 여러 개 쌓아 멀티 헤드 어텐션 층을 구현했다. MultiHeadAttention 클래스는 멀티 헤드 층으로 시작한 다음 이 층을 내부에서 개별 어텐션 헤드로 나눈다.

2개의 어첸션 헤드를 사용하는 MultiHeadAttentionWrapper 클래스에서는 2개의 가중치 행렬 $\mathbf{X_{q1}}$과 $\mathbf{X_{q2}}$를 초기화한 후 2개의 쿼리 행렬 $\mathbf{Q_{1}}$과 $\mathbf{Q_{2}}$를 계산한다. MultiHeadAttention 클래스에서는 큰 가중치 행렬 $\mathbf{W_{q}}$ 하나를 초기화한 다음 입력과 행렬 곱셈을 한 번 수행하여 쿼리 행렬 $\mathbf{Q}$를 얻는다. 그 다음 쿼리 행렬을 $\mathbf{Q_1}$과 $\mathbf{Q_2}$로 나눈다.

파이토치의 .view와 .transpose 메서드를 사용해 텐서 크기를 변경하고 전치하여 쿼리, 키, 값 텐서를 분할한다. 먼저 입력을 (쿼리, 키, 값을 위한 선형 층을 통해) 변환한 다음 여러 개의 헤드를 나타내도록 크기를 변경한다.

d_out 차원을 num_heads와 head_dim으로 나누는 것이 핵심이다. 여기에서 head_dim = d_out / num_heads이다. 이 분할을 .view 메서드를 사용해 수행된다. 이를 통해 (b, num_tokens, d_out) 차원의 텐서를 (b, num_tokens, num_heads, head_dim) 차원으로 변환한다.

그런 다음 텐서를 전치하여 num_heads 차원을 num_tokens 차원 앞으로 이동시켜 (b, num_heads, num_tokens, head_dim) 크기를 만든다.

MultiHeadAttention에서 어텐션 가중치와 문맥 벡터를 계산한 후 모든 헤드의 문맥 벡터를 전치하여 (b, num_tokens, num_heads, head_dim) 크기로 되돌린다. 그런 다음 이 벡터의 크기를 변경하여(펼쳐서) (b, num_tokens, d_out) 크기로 만든다. 결과적으로 모든 헤드의 출력을 결합한 효과를 낸다.

또한 MultiHeadAttention에서 헤드를 결합한 후 CausalAttention 클래스에 없던 출력용 투영층(self.proj)를 추가했다.

In [ ]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)